# Nvidia Nemotron Reasoning Challenge
## Score: .64

In [ ]:
import subprocess, sys, os
from pathlib import Path
def resolve_python_path(target_dir):
    for pth_file in Path(target_dir).glob("*.pth"):
        with pth_file.open() as fp:
            relpath = fp.read()
            rel_pack_path = (pth_file.parent/relpath)
            if rel_pack_path.exists():
                print(f"append {rel_pack_path}")
                sys.path.append(str(rel_pack_path))


offline_dir = "/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"
target_dir = "/kaggle/working/packages"

os.makedirs(target_dir, exist_ok=True)
resolve_python_path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/")

if os.path.exists(offline_dir):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index",
        "--find-links", offline_dir,
        "--target", target_dir,
        "datasets", "trl"
    ])
    print("Installed from offline packages")

sys.path.append(target_dir)
resolve_python_path(target_dir)

import datasets, cutlass

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys, stat, shutil, gc, zipfile, re, random
import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU detected. In Kaggle, set Accelerator to GPU before running training.")
print(f"torch={torch.__version__} | cuda={torch.cuda.is_available()} | gpu={torch.cuda.get_device_name(0)}")


In [ ]:
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast: x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None: out = out + bias.float()
    if z is not None: out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    print("Triton ptxas fix applied.")

In [ ]:
SEED = 42
SAMPLE_SIZE = 1800
MAX_SEQ_LEN = 2048
NUM_EPOCHS = 1.0
BATCH_SIZE = 1
GRAD_ACCUM = 8
LR = 8e-5
WARMUP_RATIO = 0.08
WEIGHT_DECAY = 0.01
LORA_RANK = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

RUN_NAME = f"run_new_multiview_masked_s{SAMPLE_SIZE}_e{NUM_EPOCHS}"
OUTPUT_DIR = f"/kaggle/working/adapter_{RUN_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

_MODEL_CANDIDATES = [
    "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1",
    "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default",
]
MODEL_PATH = None
for _mp in _MODEL_CANDIDATES:
    if os.path.isdir(_mp):
        MODEL_PATH = _mp
        print(f"Using local model path: {MODEL_PATH}")
        break
if MODEL_PATH is None:
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Using kagglehub model path: {MODEL_PATH}")

OFFICIAL_TRAIN_PATH = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"
if not os.path.isfile(OFFICIAL_TRAIN_PATH):
    raise FileNotFoundError(f"Missing official train.csv: {OFFICIAL_TRAIN_PATH}")

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

train_df = pl.read_csv(OFFICIAL_TRAIN_PATH)
train_df = train_df.with_columns(
    pl.col("prompt").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars(),
    pl.col("answer").cast(pl.Utf8, strict=False).fill_null("").str.strip_chars(),
)
train_df = train_df.filter(
    pl.col("prompt").str.len_chars() > 0,
    pl.col("answer").str.len_chars() > 0,
)
if len(train_df) > SAMPLE_SIZE:
    train_df = train_df.sample(n=SAMPLE_SIZE, seed=SEED)

base_rows = train_df.to_dicts()
hints = [
    "\nSolve carefully and put the final answer in \\boxed{}.",
    "\nDouble-check your work, then output only the final answer in \\boxed{}.",
    "\nThink briefly, verify, and end with \\boxed{}.",
]

views = []
for row in base_rows:
    prompt = row["prompt"]
    answer = row["answer"]

    # View A: strict metric-shaped target.
    views.append({
        "user": prompt + random.choice(hints),
        "assistant": f"\\boxed{{{answer}}}",
    })

    # View B: short verification template.
    views.append({
        "user": prompt + "\nInclude a short verification before final answer.",
        "assistant": "Check: verify the key constraint and arithmetic once.\n" + f"\\boxed{{{answer}}}",
    })

    # View C: hard negative for integer answers only.
    if re.fullmatch(r"-?\d+", answer):
        v = int(answer)
        wrong = str(v + (1 if v != -1 else 2))
        views.append({
            "user": prompt + f"\nA candidate answer is {wrong}. If incorrect, reject it and give the correct final answer in \\boxed{{}}.",
            "assistant": f"The candidate {wrong} is incorrect.\\n\\boxed{{{answer}}}",
        })

raw_dataset = Dataset.from_list(views)
print(f"Official rows used: {len(base_rows)} | Training views: {len(raw_dataset)}")
print(f"Run: {RUN_NAME} | output: {OUTPUT_DIR}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def apply_template(user_msg, assistant_msg):
    try:
        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except Exception:
        return (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
        )

def format_chat_for_mask(user_text, assistant_text):
    prefix = f"<|im_start|>user\n{user_text}<|im_end|>\n<|im_start|>assistant\n"
    full = f"{prefix}{assistant_text}<|im_end|>"
    return prefix, full

def tokenize_with_assistant_only_loss(examples):
    out_input_ids, out_mask, out_labels = [], [], []
    for user_text, assistant_text in zip(examples["user"], examples["assistant"]):
        prefix, full = format_chat_for_mask(user_text, assistant_text)
        full_enc = tokenizer(
            full,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            padding="max_length",
        )
        prefix_enc = tokenizer(
            prefix,
            truncation=True,
            max_length=MAX_SEQ_LEN,
            padding=False,
        )
        prefix_len = min(len(prefix_enc["input_ids"]), MAX_SEQ_LEN)

        labels = full_enc["input_ids"][:]
        for i in range(prefix_len):
            labels[i] = -100
        for i, m in enumerate(full_enc["attention_mask"]):
            if m == 0:
                labels[i] = -100

        out_input_ids.append(full_enc["input_ids"])
        out_mask.append(full_enc["attention_mask"])
        out_labels.append(labels)

    return {
        "input_ids": out_input_ids,
        "attention_mask": out_mask,
        "labels": out_labels,
    }

tokenized_dataset = raw_dataset.map(
    tokenize_with_assistant_only_loss,
    batched=True,
    remove_columns=raw_dataset.column_names,
)
print(f"Tokenized rows: {len(tokenized_dataset)}")

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map={"": 0},
    trust_remote_code=True,
    dtype=torch.bfloat16,
)
model.gradient_checkpointing_enable()

for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    target_modules="all-linear",
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

train_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    weight_decay=WEIGHT_DECAY,
    bf16=True,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    max_grad_norm=1.0,
)

def collate(features):
    return {
        "input_ids": torch.tensor([f["input_ids"] for f in features], dtype=torch.long),
        "attention_mask": torch.tensor([f["attention_mask"] for f in features], dtype=torch.long),
        "labels": torch.tensor([f["labels"] for f in features], dtype=torch.long),
    }

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=tokenized_dataset,
    data_collator=collate,
)

print("Starting masked-loss training...")
trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)

print(f"Adapter saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

In [ ]:
# Pre-submit sanity checks (run before clicking Submit)
import os

print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("WARNING: GPU is disabled. Training cells will fail before creating submission.zip")

required_adapter_files = ["adapter_config.json", "adapter_model.safetensors"]
missing = [f for f in required_adapter_files if not os.path.isfile(os.path.join(OUTPUT_DIR, f))]
if missing:
    raise FileNotFoundError(
        f"Missing adapter files in {OUTPUT_DIR}: {missing}. Run training cell first."
    )

print("Adapter files exist:")
for f in required_adapter_files:
    p = os.path.join(OUTPUT_DIR, f)
    print(f"  {p} ({os.path.getsize(p)/1024/1024:.2f} MB)")

zip_path = "/kaggle/working/submission.zip"
if os.path.isfile(zip_path):
    print(f"Found existing {zip_path} ({os.path.getsize(zip_path)/1024/1024:.2f} MB)")
else:
    print(f"No existing {zip_path} yet. Run packaging cell next.")

In [ ]:
import zipfile
import os

zip_path = "/kaggle/working/submission.zip"

print(f"Packaging files from {OUTPUT_DIR}...")


REQUIRED_ZIP = {"adapter_config.json", "adapter_model.safetensors"}
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in REQUIRED_ZIP:
        fpath = os.path.join(OUTPUT_DIR, fname)
        if not os.path.isfile(fpath):
            raise FileNotFoundError(f"Missing {fpath}")
        zf.write(fpath, arcname=fname)

print(f"Created {zip_path} ({os.path.getsize(zip_path) / 1024 / 1024:.1f} MB)")

with zipfile.ZipFile(zip_path, 'r') as zf:
    zip_contents = zf.namelist()
    print(f"Zip Contents: {zip_contents}")
    
    if "adapter_config.json" not in zip_contents:
        raise AssertionError("CRITICAL ERROR: adapter_config.json is missing from the zip. The Kaggle evaluation will fail.")
    if "adapter_model.safetensors" not in zip_contents:
         raise AssertionError("CRITICAL ERROR: adapter_model.safetensors is missing from the zip.")

print("submission.zip is ready")